# Credit Card Fraud Detection – Cost-Sensitive ML Modeling
### Author: Raja
### Objective: Minimize financial loss in highly imbalanced fraud dataset

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score,precision_recall_curve,average_precision_score,roc_auc_score,roc_curve,confusion_matrix
SEED = 18
np.random.seed(SEED)
random.seed(SEED)

NameError: name 'random' is not defined

In [ ]:
df=pd.read_csv(r"C:\Users\pesak\Downloads\archive\creditcard.csv")
df = df.astype("float32")
#print(df.head())
#print(df.shape)
#print(df['Class'].value_counts())
print("Fraud %:", df["Class"].mean() * 100)



In [ ]:
X=df.drop("Class",axis=1)
Y=df["Class"]

x_train,x_temp,y_train,y_temp=train_test_split(X,Y,test_size=0.3,random_state=18,stratify=Y)
x_val,x_test,y_val,y_test=train_test_split(x_temp,y_temp,test_size=0.5,random_state=18,stratify=y_temp)


In [ ]:
dm=DummyClassifier(strategy="most_frequent")
dm.fit(x_train,y_train)
dm_pred=dm.predict(x_test)
dm_prob=dm.predict_proba(x_test)[:,1]

base_recall=recall_score(y_test,dm_pred)
base_roc=roc_auc_score(y_test,dm_prob)
base_pr=average_precision_score(y_test,dm_prob)
base_precision=precision_score(y_test,dm_pred)
print("BASE RECALL",base_recall)
print("BASE PRECISION",base_precision)
print("BASE ROC:",base_roc)
print("BASE PR:",base_pr)

In [ ]:
lg_pipe=Pipeline([
    ("scaler",StandardScaler()),
    ("model",LogisticRegression(class_weight="balanced",solver="liblinear",max_iter=1000,random_state=18))
])
lg_param_grid={
    "model__C":[0.1,1,10,],
    "model__penalty":["l1","l2"]
}
lg_grid=GridSearchCV(
    lg_pipe,
    param_grid=lg_param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=2
)
lg_grid.fit(x_train,y_train)
lg_grid_prob=lg_grid.predict_proba(x_val)[:,1]
lg_grid_pred=lg_grid.predict(x_val)
print(lg_grid.best_params_)

#threshold tuning

precision,recall,threshold=precision_recall_curve(y_val,lg_grid_prob)
res_lg=[]
for pr,re,th in zip(precision[:-1],recall[:-1],threshold):
    f1=(2*(pr*re)/(pr+re)) if (pr+re)!=0 else 0
    res_lg.append((f1,pr,re,th))
thres=max(res_lg,key=lambda x:x[0])
best_thres=thres[3]
#final evalution

y_prob=lg_grid.predict_proba(x_test)[:,1]
y_pred=(y_prob>best_thres).astype(int)

lg_roc=roc_auc_score(y_test,y_prob)
lg_pr=average_precision_score(y_test,y_prob)
lg_recall=recall_score(y_test,y_pred)
lg_precision=precision_score(y_test,y_pred)
print("LG RECALL:",lg_recall)
print("LG_PRECISION:",lg_precision)
print("LG_ROC:",lg_roc)
print("LG_PR:",lg_pr)
print("Confusion_matrix:",confusion_matrix(y_test,y_pred))
print("Logistic training finished")
#graphing

precision,recall,_=precision_recall_curve(y_test,y_prob)
plt.plot(recall,precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("PRC Curve")
plt.show()

fpr,tpr,_=roc_curve(y_test,y_prob)
plt.plot(fpr,tpr)
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve")
plt.show()

#lg_permutation

lg_permutation=permutation_importance(
    lg_grid.best_estimator_,
    x_test,
    y_test,
    random_state=18,
    n_repeats=3,
    scoring="average_precision",
    n_jobs=2
)
lg_importance=pd.DataFrame({
    "Features":x_test.columns,
    "Importance":lg_permutation.importances_mean
}).sort_values(by="Importance",ascending=False)

print(lg_importance.head(10))



In [ ]:
print("Starting Random Forest training...")
rf_pipe=Pipeline([
    ("model",RandomForestClassifier(class_weight="balanced",bootstrap=True,max_features="sqrt",n_jobs=2))
])
rf_param_grid={
    "model__n_estimators":[50,100],
    "model__max_depth":[3,5]
}
rf_grid=GridSearchCV(
    rf_pipe,
    param_grid=rf_param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=2
)
rf_grid.fit(x_train,y_train)
rf_prob=rf_grid.predict_proba(x_val)[:,1]
precision,recall,threshold=precision_recall_curve(y_val,rf_prob)
res_rf=[]
for pr,re,th in zip(precision[:-1],recall[:-1],threshold):
    f1=(2*(pr*re)/(pr+re)) if (pr+re)!=0 else 0
    res_rf.append((f1,pr,re,th))
thres=max(res_rf,key=lambda x:x[0])
rf_best_thres=thres[3]
        
prob=rf_grid.predict_proba(x_test)[:,1]
pred=(prob>rf_best_thres).astype(int)
rf_roc=roc_auc_score(y_test,prob)
rf_pr=average_precision_score(y_test,prob)
rf_recall=recall_score(y_test,pred)
rf_precision=precision_score(y_test,pred)
print("RF RECALL:",rf_recall)
print("RF PRECISION:",rf_precision)
print("RF ROC:",rf_roc)
print("RF PR:",rf_pr)

precision,recall,_=precision_recall_curve(y_test,prob)
plt.plot(recall,precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("PRC Curve")
plt.show()

fpr,tpr,_=roc_curve(y_test,prob)
plt.plot(fpr,tpr)
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve")
plt.show()

rf_permutation=permutation_importance(
    rf_grid.best_estimator_,
    x_test,
    y_test,
    n_repeats=3,
    n_jobs=2,
    scoring="average_precision"
)
rf_importance=pd.DataFrame({
    "Features":x_test.columns,
    "Importance":rf_permutation.importances_mean
}).sort_values(by="Importance",ascending=False)

print(rf_importance.head(10))


In [ ]:
smote = SMOTE(random_state=18)
lg_smt_pipe=ImbPipeline([
    ("smote",smote),
    ("model",LogisticRegression(max_iter=1000,solver="liblinear",random_state=18))
])
lg_smt_parm={
    "model__C":[0.1,1,10],
    "model__penalty":["l1","l2"]
}
lg_smt_grid=GridSearchCV(
    lg_smt_pipe,
    param_grid=lg_smt_parm,
    cv=3,
    scoring="roc_auc",
    n_jobs=2
)
lg_smt_grid.fit(x_train,y_train)
lg_smt_prob_v=lg_smt_grid.predict_proba(x_val)[:,1]
precision,recall,threshold=precision_recall_curve(y_val,lg_smt_prob_v)
res_lg_smt=[]
for pr,re,th in zip(precision[:-1],recall[:-1],threshold):
    f1=(2*(pr*re)/(pr+re)) if (pr+re)!=0 else 0
    res_lg_smt.append((f1,pr,re,th))
thres=max(res_lg_smt,key=lambda x:x[0])
lg_smt_thres=thres[3]
lg_smt_prob=lg_smt_grid.predict_proba(x_test)[:,1]
lg_smt_pred=(lg_smt_prob>lg_smt_thres).astype(int)
lg_smt_recall=recall_score(y_test,lg_smt_pred)
lg_smt_precision=precision_score(y_test,lg_smt_pred)
lg_smt_roc=roc_auc_score(y_test,lg_smt_prob)
lg_smt_pr=average_precision_score(y_test,lg_smt_prob)
print("LG SMT RECALL:",lg_smt_recall)
print("LG SMT PRECISION:",lg_smt_precision)
print("LG SMT ROC:",lg_smt_roc)
print("LG SMT PR:",lg_smt_pr)


In [ ]:
rf_smt_pipe=ImbPipeline([
    ("smote",smote),
    ("model",RandomForestClassifier(bootstrap=True,max_features='sqrt',random_state=18,n_jobs=2))
])
rf_smt_parm={
    "model__max_depth":[3,5],
    "model__n_estimators":[50,100]
}
rf_smt_grid=GridSearchCV(
    rf_smt_pipe,
    param_grid=rf_smt_parm,
    cv=3,
    n_jobs=2,
    scoring="roc_auc"
)
rf_smt_grid.fit(x_train,y_train)
rf_smt_prob_v=rf_smt_grid.predict_proba(x_val)[:,1]

precision,recall,threshold=precision_recall_curve(y_val,rf_smt_prob_v)

res_rf_smt=[]
for pr,re,th in zip(precision[:-1],recall[:-1],threshold):
    f1=(2*(pr*re)/(pr+re)) if (pr+re)!=0 else 0
    res_rf_smt.append((f1,pr,re,th))
thres=max(res_rf_smt,key=lambda x:x[0])
rf_smt_thres=thres[3]

rf_smt_prob=rf_smt_grid.predict_proba(x_test)[:,1]
rf_smt_pred=(rf_smt_prob>rf_smt_thres).astype(int)
rf_smt_recall=recall_score(y_test,rf_smt_pred)
rf_smt_precision=precision_score(y_test,rf_smt_pred)
rf_smt_roc=roc_auc_score(y_test,rf_smt_prob)
rf_smt_pr=average_precision_score(y_test,rf_smt_prob)
print("RF SMT RECALL:",rf_smt_recall)
print("RF SMT PRECISION:",rf_smt_precision)
print("RF SMT ROC:",rf_smt_roc)
print("RF SMT PR:",rf_smt_pr)


In [ ]:
results = pd.DataFrame([
    ["Dummy", base_roc, base_pr, base_recall, base_precision],
    ["Logistic Balanced", lg_roc, lg_pr, lg_recall, lg_precision],
    ["RF Balanced", rf_roc, rf_pr, rf_recall, rf_precision],
    ["Logistic SMOTE", lg_smt_roc, lg_smt_pr, lg_smt_recall, lg_smt_precision],
    ["RF SMOTE", rf_smt_roc, rf_smt_pr, rf_smt_recall, rf_smt_precision],
], columns=["Model", "ROC_AUC", "PR_AUC", "Recall", "Precision"])

print(results.sort_values(by="PR_AUC", ascending=False))


In [ ]:
def compute_cost(y_true, y_pred, fp_cost=50, fn_cost=10000):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fp * fp_cost + fn * fn_cost

rf_cost = compute_cost(y_test, pred)
lg_cost = compute_cost(y_test, y_pred)
lg_smt_cost = compute_cost(y_test, lg_smt_pred)
rf_smt_cost = compute_cost(y_test, rf_smt_pred)

print("RF Cost:", rf_cost)
print("LG Cost:", lg_cost)
print("LG SMOTE Cost:", lg_smt_cost)
print("RF SMOTE Cost:", rf_smt_cost)

## ✅ Final Deployment Model

After comparing all models using PR-AUC and cost-sensitive evaluation, 
**Logistic Regression (class_weight="balanced")** was selected as the final deployment model.

### 📊 Why?

- It achieved strong PR-AUC (0.69)
- Maintained high precision (83.5%)
- Achieved solid recall (75.7%)
- Produced the **lowest expected financial cost (180,550)**
- Simpler and more interpretable than Random Forest

This model provides the best trade-off between fraud detection performance and business impact.

## 🔍 Key Insights

- PR-AUC is more informative than ROC-AUC for highly imbalanced datasets (0.17% fraud rate).
- Threshold tuning significantly improved model performance compared to default 0.5.
- SMOTE slightly improved Logistic Regression but did not significantly benefit Random Forest.
- Cost-sensitive evaluation changed the final model selection.
- Logistic Regression was selected for deployment due to lowest expected financial loss and better interpretability.